##Problem 1 - Logistic regression on 1D data without using the built-in function to perform binary classification.

In [1]:
import numpy as np

In [2]:
X = np.random.rand(100,1)
noise = 0.5 * np.random.rand(100,1)
score = X + noise
y = (score > 0.5).astype(int)

In [3]:
X[:10], y[:10]

(array([[0.91375621],
        [0.82621387],
        [0.06640775],
        [0.23725013],
        [0.42420961],
        [0.09243411],
        [0.4008057 ],
        [0.19371795],
        [0.09952389],
        [0.86990639]]),
 array([[1],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [0],
        [1]]))

In [4]:
X.shape, y.shape

((100, 1), (100, 1))

In [5]:
y1d = y.ravel()
y1d.shape

(100,)

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X , y1d, test_size=0.3, random_state=42)

In [8]:
def sigmoid(z):
  return 1/(1 + np.exp(-z))

def predict(Xb, theta):
  return sigmoid(Xb @ theta)

In [9]:
def gradients(Xb,y,y_pred):
  error = y_pred - y.reshape(-1,1)
  grad = (1/len(Xb)) * Xb.T @ error
  return grad

In [10]:
def train_logistics(X, y ,lr=0.1, epochs=1000):
  Xb = np.c_[np.ones((len(X),1)), X]
  theta = np.zeros((Xb.shape[1],1))
  for _ in range(epochs):
    y_pred = predict(Xb,theta)
    grad = gradients(Xb,y,y_pred)
    theta -= lr * grad
  return theta

In [11]:
theta = train_logistics(X_train,y_train, lr=0.1, epochs=1000)

In [12]:
from sklearn.metrics import accuracy_score

In [13]:
Xb_train = np.c_[np.ones((len(X_train),1)),X_train]
train_preds = (predict(Xb_train,theta) >= 0.5).astype(int)
train_acc = accuracy_score(y_train, train_preds)

In [14]:
Xb_test = np.c_[np.ones((len(X_test),1)),X_test]
test_preds = (predict(Xb_test,theta) >= 0.5).astype(int)
test_acc = accuracy_score(y_test, test_preds)

In [15]:
print("Model parameters : ",theta.ravel())
print(f"Training accuracy : {train_acc*100:.2f}%")
print(f"Test accuracy : {test_acc*100:.2f}%")

Model parameters :  [-1.32500406  4.56065229]
Training accuracy : 82.86%
Test accuracy : 93.33%


##Problem 2 - Logistic regression on 2D data using the sklearn built-in function to perform binary classification.


In [16]:
X = np.random.rand(100,2)
score = X[:,0] + X[:,1] + 0.2*np.random.randn(200)
y = (score > 0.5).astype(int)

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X, y ,test_size=0.3, random_state=42)

In [19]:
from sklearn.linear_model import LogisticRegression

In [20]:
model = LogisticRegression(max_iter=1000)

In [21]:
model.fit(X_train,y_train)

LogisticRegression(max_iter=1000)

In [22]:
print("Intercept (bias): ",model.intercept_)
print("Coefficient (weight): ",model.coef_)

Intercept (bias):  [-0.60330368]
Coefficient (weight):  [[2.53677138 2.92401242]]


In [23]:
train_preds = model.predict(X_train)
train_acc = accuracy_score(y_train, train_preds)
print(f"Training accuracy : {train_acc*100:.2f}%")

Training accuracy : 87.86%


In [24]:
test_preds = model.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)
print(f"Test accuracy : {test_acc*100:.2f}%")

Test accuracy : 88.33%


##Problem 3 - Effect of standard scaling on 2D data to perform binary classification using the sklearn built-in function.

In [25]:
X1 = np.random.rand(100,1)
X2 = np.random.rand(100,1)
X = np.hstack([X1, X2])
score = 0.3*X1 + 0.7*X2 + 0.2*np.random.randn(100,1)
y = (score > 0.5).astype(int).ravel()

In [27]:
print(X1.min(),X1.max(),X2.min(),X2.max())

0.0015253009349807112 0.9837796908342334 0.017115814036484545 0.9893605554741309


In [28]:
X_train,X_test,y_train,y_test = train_test_split(X, y ,test_size=0.3, random_state=42)

In [29]:
model_unscaled = LogisticRegression(max_iter=1000)
model_unscaled.fit(X_train,y_train)

LogisticRegression(max_iter=1000)

In [31]:
y_pred_unscaled = model_unscaled.predict(X_test)
acc_unscaled = accuracy_score(y_test, y_pred_unscaled)
print(f"Intercept : ", model_unscaled.intercept_)
print(f"Coefficient : ", model_unscaled.coef_)
print(f"Accuracy without scaling : {acc_unscaled*100:.2f}%")
print(f"Coefficient ratio |β1/β2| : ", abs(model_unscaled.coef_[0,0])/abs(model_unscaled.coef_[0,1]))

Intercept :  [-1.14715585]
Coefficient :  [[0.668566   2.41606475]]
Accuracy without scaling : 80.00%
Coefficient ratio |β1/β2| :  0.2767169214870069


In [32]:
from sklearn.preprocessing import StandardScaler

In [33]:
scaler = StandardScaler()

In [34]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [35]:
model_scaled = LogisticRegression(max_iter=1000)
model_scaled.fit(X_train_scaled,y_train)

LogisticRegression(max_iter=1000)

In [36]:
y_pred_scaled = model_unscaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)
print(f"Intercept : ", model_scaled.intercept_)
print(f"Coefficient : ", model_scaled.coef_)
print(f"Accuracy without scaling : {acc_scaled*100:.2f}%")
print(f"Coefficient ratio |β1/β2| : ", abs(model_scaled.coef_[0,0])/abs(model_scaled.coef_[0,1]))

Intercept :  [0.71487989]
Coefficient :  [[0.33013879 1.31933199]]
Accuracy without scaling : 73.33%
Coefficient ratio |β1/β2| :  0.2502317749572872


##Problem 4 - Effect of regularization on 2D (circular data) to perform binary classification using the sklearn built-in function.

In [37]:
from sklearn.datasets import make_circles

In [38]:
X, y = make_circles(n_samples=500, noise=0.5)

In [39]:
X_train,X_test,y_train,y_test = train_test_split(X, y ,test_size=0.3, random_state=42)

In [40]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [41]:
from sklearn.preprocessing import PolynomialFeatures

In [48]:
poly2 = PolynomialFeatures(degree=2)
X_train_poly2 = poly2.fit_transform(X_train_scaled)
X_test_poly2 = poly2.transform(X_test_scaled)

model_poly2 = LogisticRegression(max_iter=1000, penalty=None)
model_poly2.fit(X_train_poly2, y_train)

train_acc_poly2 = accuracy_score(y_train, model_poly2.predict(X_train_poly2))
test_acc_poly2 = accuracy_score(y_test, model_poly2.predict(X_test_poly2))

print(f"Training accuracy : {train_acc_poly2 * 100:.2f}%")
print(f"Test accuracy : {test_acc_poly2 * 100:.2f}%")
print("Intercept : ", model_poly2.intercept_)
print("Coefficient : ", model_poly2.coef_)
print("Coefficient ratio |β1/β2| : ", abs(model_poly2.coef_[0,0])/abs(model_poly2.coef_[0,1]))

Training accuracy : 56.57%
Test accuracy : 58.00%
Intercept :  [0.16452797]
Coefficient :  [[ 0.16452797 -0.05839179  0.03064064 -0.18148798  0.08013063 -0.21311359]]
Coefficient ratio |β1/β2| :  2.817655709918482


In [49]:
poly10 = PolynomialFeatures(degree=10)
X_train_poly10 = poly10.fit_transform(X_train_scaled)
X_test_poly10 = poly10.transform(X_test_scaled)

model_poly10 = LogisticRegression(max_iter=1000, penalty=None)
model_poly10.fit(X_train_poly10, y_train)

train_acc_poly10 = accuracy_score(y_train, model_poly10.predict(X_train_poly10))
test_acc_poly10 = accuracy_score(y_test, model_poly10.predict(X_test_poly10))

print(f"Training accuracy : {train_acc_poly10 * 100:.2f}%")
print(f"Test accuracy : {test_acc_poly10 * 100:.2f}%")
print("Intercept : ", model_poly10.intercept_)
print("Coefficient : ", model_poly10.coef_)
print("Coefficient ratio |β1/β2| : ", abs(model_poly10.coef_[0,0])/abs(model_poly10.coef_[0,1]))

Training accuracy : 62.86%
Test accuracy : 56.00%
Intercept :  [0.10685367]
Coefficient :  [[ 0.10685367 -0.44489828 -0.13338342  0.13670313 -0.19304993  0.11924936
   0.28099628 -0.04524277 -0.13771702  0.52635399  0.10628928  0.20085837
   0.02045022  0.34319422 -0.24375028  0.42054768 -0.10333758 -0.30187743
   0.14769533  0.23711517  0.44541205 -0.3524582   0.04070881  0.12894665
   0.30014091 -0.08267398  0.20714855 -0.17784486 -0.11834652  0.20703076
  -0.58507492 -0.53970968  0.17493093  0.21176654  0.0871778  -0.58908752
   0.20545976 -0.49983313 -0.37333002  0.03782733  0.66838368  0.00287524
  -0.34776003 -0.23832464  0.22090091 -0.01720441 -0.03799707  0.25724027
   0.01477437 -0.17268932  0.05926741  0.1563958  -0.03060257 -0.08629605
   0.12006181 -0.03862289  0.15625325  0.09659695 -0.18549603 -0.14623879
   0.37008049 -0.07681844 -0.23581551  0.10230044  0.07204693 -0.04693473]]
Coefficient ratio |β1/β2| :  0.240175502880804


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [50]:
model_poly10_reg = LogisticRegression(max_iter=1000, penalty='l2', C=0.1)
model_poly10_reg.fit(X_train_poly10, y_train)

train_acc_poly10_reg = accuracy_score(y_train, model_poly10_reg.predict(X_train_poly10))
test_acc_poly10_reg = accuracy_score(y_test, model_poly10_reg.predict(X_test_poly10))

print(f"Training accuracy : {train_acc_poly10_reg * 100:.2f}%")
print(f"Test accuracy : {test_acc_poly10_reg * 100:.2f}%")
print("Intercept : ", model_poly10_reg.intercept_)
print("Coefficient : ", model_poly10_reg.coef_)
print("Coefficient ratio |β1/β2| : ", abs(model_poly10_reg.coef_[0,0])/abs(model_poly10_reg.coef_[0,1]))

Training accuracy : 63.71%
Test accuracy : 53.33%
Intercept :  [0.16149775]
Coefficient :  [[ 0.03835415 -0.10157555  0.10731231  0.03540898 -0.00183961 -0.01673782
   0.09086884  0.02093864 -0.05879409  0.1994268   0.01595087  0.03498417
  -0.01587775  0.11786521 -0.09461961  0.13046429 -0.0379468  -0.08603702
   0.04828924  0.02767418  0.10215575 -0.08612544 -0.05600878 -0.00225503
   0.08484939 -0.01746301  0.07647896 -0.05830564 -0.02634072 -0.01137901
  -0.1548021  -0.13287558  0.03452566  0.03905044 -0.00321699 -0.22441899
   0.05918916 -0.22463358 -0.10919972  0.01511929  0.13496093  0.02950635
  -0.04322907 -0.06759099  0.07652724 -0.01131461 -0.00240036  0.06719856
   0.0570909  -0.01119266 -0.07535339  0.04573075  0.03061156 -0.02298939
   0.04913004 -0.01387802  0.07647429  0.03367651 -0.08162909 -0.03715849
   0.19197364 -0.03633885 -0.12816556  0.02824226  0.02530983 -0.01743659]]
Coefficient ratio |β1/β2| :  0.3775922867258393


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
